In [1]:
import argparse
import os
from pathlib import Path
from omegaconf import OmegaConf
from itertools import chain
from toolz import identity, pipe
from toolz.curried import map as map_c

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, ConcatDataset
from lightning import LightningModule, Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import MLFlowLogger
# from torchmetrics.regression import ...
from torchdiffeq import odeint # odeint_adjoint as odeint
from filterpy.kalman import UnscentedKalmanFilter, MerweScaledSigmaPoints

import plotly.graph_objects as go

from experiment.motion_sense.utils.dataset import TrajectoryDataset
from experiment.motion_sense.utils.field import FieldLitModule, Field
from experiment.motion_sense.utils.metric import SmoothedTrajectoryLoss

import mlflow

/Users/sem-k32/10_sem/n_ode/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOISE_SIGMA = 1e-2
act = "jog"
subj = 0

In [ ]:
config = OmegaConf.load("/Users/sem-k32/10_sem/n_ode/experiment/motion_sense/config.yaml")

In [4]:
state_noise_sigma = pd.read_csv(
    os.path.join(config.field_dir, act, str(subj), "traj_std.csv"),
    index_col=0
).to_numpy().flatten()
state_names = pipe(
    config.data_types,
    map_c(lambda name: [name + '.' + suffix for suffix in ['x', 'y', 'z']]),
    chain.from_iterable,
    list
)
state_dim = 6

In [8]:
field_lit_adapter = FieldLitModule.load_from_checkpoint(
    "/Users/sem-k32/10_sem/n_ode/fields/jog/0/best.ckpt",
    d=state_dim, dt=config.dt, state_noise_sigma=state_noise_sigma,
    noise_sigma=NOISE_SIGMA, state_names=state_names
)

In [12]:
field_lit_adapter.field.nonlinear_add_scale

Parameter containing:
tensor([-0.0208], requires_grad=True)

In [11]:
torch.linalg.eigvals(field_lit_adapter.field.A.weight)

tensor([-0.0615+0.0725j, -0.0615-0.0725j, -0.0095+0.0000j, -0.0104+0.0000j,
        -0.0401+0.0336j, -0.0401-0.0336j], grad_fn=<LinalgEigBackward0>)